# Football Detection + Stable Tracking + Team Separation — Working REST Version

Bu sürüm, önceki notebook'taki `inference_sdk / supervision / Pillow` çakışmasını tamamen devre dışı bırakır. Roboflow modeline **direkt REST API** ile istek atar, videonun her frame'ini çıktı videosuna yazar ve oyuncuların altında elips gösterir.

Özellikler:
- `football-players-detection-3zvbc/11` Roboflow modelini kullanır.
- `ball / player / goalkeeper / referee` sınıflarını işler.
- Oyuncu ve kaleciler için basit ama stabil IoU + mesafe tabanlı tracking yapar.
- Oyuncu ID'lerini mümkün olduğunca korur.
- Oyuncuların altında elips çizer.
- Top için üçgen işaret çizer.
- Forma rengine göre otomatik `team_1 / team_2` ayrımı yapar.
- Sonuç videosu ve CSV dosyası üretir.

> Not: En doğru ve akıcı sonuç için `DETECT_EVERY_N_FRAMES = 1` bırak. Hız için 2-3 yapılabilir ama tespitler daha seyrek güncellenir.


In [ ]:
# ============================================================
# 0) Kurulum
# ============================================================
# Bu notebook inference_sdk ve supervision kurmaz.
# Böylece önceki Pillow / supervision import hatası oluşmaz.
# requests ve Pillow sürümleri Colab uyumu için pinlendi.

!pip -q install "requests==2.32.4" "Pillow==11.3.0" "opencv-python-headless>=4.8,<4.12" "pandas>=2.0" "tqdm>=4.66" "scikit-learn>=1.3"


In [ ]:
# ============================================================
# 1) Ayarlar ve importlar
# ============================================================
from getpass import getpass
from pathlib import Path
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any
import base64
import json
import math
import os
import time

import cv2
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from IPython.display import HTML, display

try:
    from sklearn.cluster import KMeans
    SKLEARN_AVAILABLE = True
except Exception as e:
    print("scikit-learn yüklenemedi, takım ayrımı kapatılacak:", repr(e))
    KMeans = None
    SKLEARN_AVAILABLE = False

# ======================
# ROBOFLOW MODEL AYARLARI
# ======================
MODEL_ID = "football-players-detection-3zvbc/11"
API_URL = "https://serverless.roboflow.com"

# Güvenlik için API key'i notebook içine sabitlemiyoruz.
# Çalıştırınca gizli olarak girmen yeterli.
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "").strip()
if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass("Roboflow API key gir: ").strip()

# ======================
# VIDEO / PERFORMANS AYARLARI
# ======================
VIDEO_PATH = ""       # Boş bırakırsan Colab upload penceresi açılır. Örn: "/content/match.mp4"
OUTPUT_DIR = "/content/football_outputs"

DETECT_EVERY_N_FRAMES = 1   # En stabil sonuç için 1. Hız için 2 veya 3 denenebilir.
MAX_SECONDS = 30            # Tüm video için None yap. İlk testte 20-30 sn önerilir.

# Roboflow tespit eşikleri
PLAYER_CONFIDENCE = 0.25
GOALKEEPER_CONFIDENCE = 0.20
REFEREE_CONFIDENCE = 0.25
BALL_CONFIDENCE = 0.10
OVERLAP = 0.30
MAX_DETECTIONS = 300

# Büyük videolarda API payload'u küçültmek için inference sırasında frame'i yeniden boyutlandırır.
# Koordinatlar otomatik olarak orijinal boyuta geri ölçeklenir.
INFERENCE_WIDTH = 1280      # None yaparsan orijinal çözünürlük gönderilir.
JPEG_QUALITY = 85
REQUEST_TIMEOUT = 90
REQUEST_RETRIES = 2

# ======================
# TRACKING AYARLARI
# ======================
MAX_MISSED_SECONDS = 1.20
MATCH_DISTANCE_PX = 150
MATCH_IOU_THRESHOLD = 0.02
MIN_HITS_TO_SHOW = 1

# ======================
# TAKIM AYRIMI AYARLARI
# ======================
TEAM_CLASSIFICATION_ENABLED = True
CLASSIFY_GOALKEEPERS = False
TEAM_MIN_SAMPLES = 24
TEAM_MAX_SAMPLES = 500
TEAM_REFIT_EVERY = 25
TEAM_MIN_TRACK_VOTES = 3

TEAM_1_NAME = "team_1"
TEAM_2_NAME = "team_2"
UNKNOWN_TEAM_NAME = "unknown"

# OpenCV BGR renkleri
TEAM_1_DRAW_COLOR = (255, 0, 0)       # mavi
TEAM_2_DRAW_COLOR = (0, 0, 255)       # kırmızı
UNKNOWN_PLAYER_COLOR = (255, 255, 0)  # cyan
BALL_COLOR = (0, 255, 0)              # yeşil
REFEREE_COLOR = (0, 220, 255)         # sarı/turuncu
GOALKEEPER_COLOR = (255, 100, 255)    # mor/pembe
TEXT_COLOR = (255, 255, 255)

# Çizim seçenekleri
DRAW_BOXES = False
DRAW_ELLIPSES = True
DRAW_BALL_TRIANGLE = True
SHOW_CLASS_NAME = False
SHOW_CONFIDENCE = False
SHOW_TRACK_ID = True
SHOW_TEAM_LABEL = False

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Ayarlar hazır.")


In [ ]:
# ============================================================
# 2) Yardımcı fonksiyonlar: video seçimi, Roboflow REST inference, parsing
# ============================================================

def choose_video_path(video_path: str = "") -> str:
    video_path = (video_path or "").strip()
    if video_path and Path(video_path).exists():
        return video_path

    try:
        from google.colab import files
        print("Video dosyanı yükle...")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("Video yüklenmedi.")
        first_name = next(iter(uploaded.keys()))
        return str(Path("/content") / first_name)
    except Exception as e:
        print("Colab upload kullanılamadı veya dosya yüklenmedi:", repr(e))
        manual = input("Video path gir: ").strip()
        if not Path(manual).exists():
            raise FileNotFoundError(f"Video bulunamadı: {manual}")
        return manual


def resize_for_inference(frame: np.ndarray, target_width: Optional[int]) -> Tuple[np.ndarray, float, float]:
    h, w = frame.shape[:2]
    if target_width is None or target_width <= 0 or w <= target_width:
        return frame, 1.0, 1.0
    new_w = int(target_width)
    new_h = int(round(h * (new_w / w)))
    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)
    scale_x = w / new_w
    scale_y = h / new_h
    return resized, scale_x, scale_y


def encode_bgr_to_jpeg_base64(frame_bgr: np.ndarray, quality: int = 85) -> Tuple[str, bytes]:
    ok, buffer = cv2.imencode(".jpg", frame_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
    if not ok:
        raise ValueError("Frame JPEG'e çevrilemedi.")
    jpg_bytes = buffer.tobytes()
    b64 = base64.b64encode(jpg_bytes).decode("utf-8")
    return b64, jpg_bytes


class RoboflowRESTClient:
    # Inference SDK kullanmadan Roboflow Serverless REST API çağrısı yapar.
    # İlk başarılı yöntemi hafızaya alır. Böylece her frame'de farklı yöntem denenmez.
    def __init__(self, api_key: str, model_id: str, api_url: str = "https://serverless.roboflow.com"):
        if not api_key:
            raise ValueError("ROBOFLOW_API_KEY boş olamaz.")
        self.api_key = api_key
        self.model_id = model_id.strip("/")
        self.api_url = api_url.rstrip("/")
        self.legacy_url = f"{self.api_url}/{self.model_id}"
        self.detect_url = f"https://detect.roboflow.com/{self.model_id}"
        self.chosen_method = None
        self.last_error = None

    def _params(self, confidence: float, overlap: float) -> Dict[str, Any]:
        return {
            "api_key": self.api_key,
            "confidence": float(confidence),
            "overlap": float(overlap),
            "max_detections": int(MAX_DETECTIONS),
            "format": "json",
            "disable_active_learning": "true",
        }

    def _post(self, method: str, image_b64: str, jpg_bytes: bytes, confidence: float, overlap: float) -> Dict[str, Any]:
        params = self._params(confidence, overlap)

        if method == "json_image_object":
            payload = {
                "api_key": self.api_key,
                "image": {"type": "base64", "value": image_b64},
            }
            r = requests.post(self.legacy_url, params=params, json=payload, timeout=REQUEST_TIMEOUT)

        elif method == "json_image_string":
            payload = {
                "api_key": self.api_key,
                "image": image_b64,
                "image_type": "base64",
            }
            r = requests.post(self.legacy_url, params=params, json=payload, timeout=REQUEST_TIMEOUT)

        elif method == "raw_base64":
            headers = {"Content-Type": "application/x-www-form-urlencoded"}
            r = requests.post(self.legacy_url, params=params, data=image_b64, headers=headers, timeout=REQUEST_TIMEOUT)

        elif method == "files_jpeg":
            files = {"file": ("frame.jpg", jpg_bytes, "image/jpeg")}
            r = requests.post(self.legacy_url, params=params, files=files, timeout=REQUEST_TIMEOUT)

        elif method == "detect_json_image_object":
            payload = {
                "api_key": self.api_key,
                "image": {"type": "base64", "value": image_b64},
            }
            r = requests.post(self.detect_url, params=params, json=payload, timeout=REQUEST_TIMEOUT)

        elif method == "detect_raw_base64":
            headers = {"Content-Type": "application/x-www-form-urlencoded"}
            r = requests.post(self.detect_url, params=params, data=image_b64, headers=headers, timeout=REQUEST_TIMEOUT)

        elif method == "infer_object_detection":
            endpoint = f"{self.api_url}/infer/object_detection"
            payload = {
                "api_key": self.api_key,
                "model_id": self.model_id,
                "image": {"type": "base64", "value": image_b64},
                "confidence": float(confidence),
                "overlap": float(overlap),
            }
            r = requests.post(endpoint, json=payload, timeout=REQUEST_TIMEOUT)

        else:
            raise ValueError(f"Bilinmeyen request method: {method}")

        if not r.ok:
            text = r.text[:700].replace("\n", " ")
            raise RuntimeError(f"HTTP {r.status_code} | {text}")

        try:
            return r.json()
        except Exception as e:
            raise RuntimeError(f"JSON parse hatası: {repr(e)} | cevap: {r.text[:500]}")

    def infer_frame(self, frame_bgr: np.ndarray, confidence: float, overlap: float) -> Tuple[Dict[str, Any], float, float]:
        small, scale_x, scale_y = resize_for_inference(frame_bgr, INFERENCE_WIDTH)
        image_b64, jpg_bytes = encode_bgr_to_jpeg_base64(small, JPEG_QUALITY)

        methods = [
            "json_image_object",
            "json_image_string",
            "raw_base64",
            "files_jpeg",
            "detect_json_image_object",
            "detect_raw_base64",
            "infer_object_detection",
        ]
        if self.chosen_method:
            methods = [self.chosen_method]

        errors = []
        for method in methods:
            for attempt in range(REQUEST_RETRIES + 1):
                try:
                    result = self._post(method, image_b64, jpg_bytes, confidence, overlap)
                    if not isinstance(result, dict):
                        raise RuntimeError(f"Beklenmeyen response tipi: {type(result)}")
                    if self.chosen_method is None:
                        self.chosen_method = method
                        print(f"Roboflow request method seçildi: {method}")
                    return result, scale_x, scale_y
                except Exception as e:
                    err = f"{method} attempt={attempt+1}: {repr(e)}"
                    errors.append(err)
                    self.last_error = err
                    time.sleep(0.4 * (attempt + 1))

        raise RuntimeError("Roboflow inference başarısız. Son hatalar:\n" + "\n".join(errors[-8:]))


def normalize_class_name(name: str) -> str:
    s = str(name or "").strip().lower().replace("-", "_").replace(" ", "_")
    mapping = {
        "football_player": "player",
        "players": "player",
        "person": "player",
        "goal_keeper": "goalkeeper",
        "goalie": "goalkeeper",
        "ref": "referee",
        "refrees": "referee",
        "soccer_ball": "ball",
        "sports_ball": "ball",
    }
    return mapping.get(s, s)


def class_threshold(cls: str) -> float:
    cls = normalize_class_name(cls)
    if cls == "player":
        return PLAYER_CONFIDENCE
    if cls == "goalkeeper":
        return GOALKEEPER_CONFIDENCE
    if cls == "referee":
        return REFEREE_CONFIDENCE
    if cls == "ball":
        return BALL_CONFIDENCE
    return PLAYER_CONFIDENCE


def parse_roboflow_predictions(result: Dict[str, Any], scale_x: float = 1.0, scale_y: float = 1.0) -> List[Dict[str, Any]]:
    preds = result.get("predictions", [])
    parsed = []

    for p in preds:
        cls_raw = p.get("class", p.get("class_name", p.get("label", "unknown")))
        cls = normalize_class_name(cls_raw)
        conf = float(p.get("confidence", p.get("class_confidence", 0.0)) or 0.0)

        if conf < class_threshold(cls):
            continue

        # Roboflow object detection format: center x/y + width/height
        if all(k in p for k in ["x", "y", "width", "height"]):
            cx = float(p["x"]) * scale_x
            cy = float(p["y"]) * scale_y
            bw = float(p["width"]) * scale_x
            bh = float(p["height"]) * scale_y
            x1 = cx - bw / 2
            y1 = cy - bh / 2
            x2 = cx + bw / 2
            y2 = cy + bh / 2
        # Bazı response'larda xyxy gelebilir diye güvenli fallback
        elif all(k in p for k in ["x1", "y1", "x2", "y2"]):
            x1 = float(p["x1"]) * scale_x
            y1 = float(p["y1"]) * scale_y
            x2 = float(p["x2"]) * scale_x
            y2 = float(p["y2"]) * scale_y
        else:
            continue

        parsed.append({
            "class": cls,
            "class_raw": cls_raw,
            "confidence": conf,
            "bbox": np.array([x1, y1, x2, y2], dtype=np.float32),
            "detection_id": p.get("detection_id", ""),
        })

    return parsed

print("Yardımcı fonksiyonlar hazır.")


In [ ]:
# ============================================================
# 3) Tracking, takım ayrımı ve çizim fonksiyonları
# ============================================================

def bbox_area(box: np.ndarray) -> float:
    x1, y1, x2, y2 = box.astype(float)
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def bbox_iou(a: np.ndarray, b: np.ndarray) -> float:
    ax1, ay1, ax2, ay2 = a.astype(float)
    bx1, by1, bx2, by2 = b.astype(float)
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    union = bbox_area(a) + bbox_area(b) - inter
    return float(inter / union) if union > 0 else 0.0


def bbox_center(box: np.ndarray) -> Tuple[float, float]:
    x1, y1, x2, y2 = box.astype(float)
    return (x1 + x2) / 2.0, (y1 + y2) / 2.0


def foot_point(box: np.ndarray) -> Tuple[float, float]:
    x1, y1, x2, y2 = box.astype(float)
    return (x1 + x2) / 2.0, y2


def center_distance(a: np.ndarray, b: np.ndarray) -> float:
    ax, ay = bbox_center(a)
    bx, by = bbox_center(b)
    return float(math.hypot(ax - bx, ay - by))


@dataclass
class Track:
    track_id: int
    class_name: str
    bbox: np.ndarray
    confidence: float
    first_frame: int
    last_frame: int
    hits: int = 1
    missed: int = 0
    team_votes: Counter = field(default_factory=Counter)
    team_label: str = UNKNOWN_TEAM_NAME


class SimpleStableTracker:
    def __init__(self, fps: float, max_missed_seconds: float = 1.2, match_distance_px: float = 150.0):
        self.fps = max(1.0, float(fps or 25.0))
        self.max_missed = max(1, int(round(max_missed_seconds * self.fps)))
        self.match_distance_px = float(match_distance_px)
        self.next_id = 1
        self.tracks: Dict[int, Track] = {}

    def _compatible_class(self, track_cls: str, det_cls: str) -> bool:
        # Oyuncu ve kaleci birbirine benzeyebilir ama kaleci takımı yanıltmasın diye ayrı tutulur.
        if track_cls == det_cls:
            return True
        return False

    def update(self, detections: List[Dict[str, Any]], frame_idx: int) -> List[Track]:
        # Sadece insan sınıflarını track ediyoruz. Top hızlı hareket ettiği için ayrı çizilir.
        trackable = [d for d in detections if d["class"] in {"player", "goalkeeper", "referee"}]
        active_tracks = list(self.tracks.values())

        candidates = []
        for ti, tr in enumerate(active_tracks):
            for di, det in enumerate(trackable):
                if not self._compatible_class(tr.class_name, det["class"]):
                    continue
                iou = bbox_iou(tr.bbox, det["bbox"])
                dist = center_distance(tr.bbox, det["bbox"])
                if iou >= MATCH_IOU_THRESHOLD or dist <= self.match_distance_px:
                    # IoU yüksek, mesafe düşük ise daha iyi.
                    score = iou - (dist / max(1.0, self.match_distance_px)) * 0.15
                    candidates.append((score, iou, dist, ti, di))

        candidates.sort(reverse=True, key=lambda x: x[0])
        matched_tracks = set()
        matched_dets = set()

        for score, iou, dist, ti, di in candidates:
            if ti in matched_tracks or di in matched_dets:
                continue
            tr = active_tracks[ti]
            det = trackable[di]
            tr.bbox = det["bbox"].astype(np.float32)
            tr.confidence = float(det["confidence"])
            tr.class_name = det["class"]
            tr.last_frame = frame_idx
            tr.hits += 1
            tr.missed = 0
            matched_tracks.add(ti)
            matched_dets.add(di)

        # Eşleşmeyen track'leri yaşlandır
        for ti, tr in enumerate(active_tracks):
            if ti not in matched_tracks:
                tr.missed += 1

        # Eşleşmeyen detection'lardan yeni track oluştur
        for di, det in enumerate(trackable):
            if di in matched_dets:
                continue
            tid = self.next_id
            self.next_id += 1
            self.tracks[tid] = Track(
                track_id=tid,
                class_name=det["class"],
                bbox=det["bbox"].astype(np.float32),
                confidence=float(det["confidence"]),
                first_frame=frame_idx,
                last_frame=frame_idx,
            )

        # Eski track'leri sil
        to_delete = [tid for tid, tr in self.tracks.items() if tr.missed > self.max_missed]
        for tid in to_delete:
            del self.tracks[tid]

        visible = [tr for tr in self.tracks.values() if tr.hits >= MIN_HITS_TO_SHOW and tr.missed <= self.max_missed]
        visible.sort(key=lambda t: t.track_id)
        return visible


class JerseyTeamClassifier:
    def __init__(self):
        self.samples: List[np.ndarray] = []
        self.kmeans = None
        self.last_fit_count = 0
        self.enabled = TEAM_CLASSIFICATION_ENABLED and SKLEARN_AVAILABLE

    def extract_feature(self, frame: np.ndarray, bbox: np.ndarray) -> Optional[np.ndarray]:
        h, w = frame.shape[:2]
        x1, y1, x2, y2 = bbox.astype(int)
        x1 = max(0, min(w - 1, x1)); x2 = max(0, min(w, x2))
        y1 = max(0, min(h - 1, y1)); y2 = max(0, min(h, y2))
        if x2 <= x1 or y2 <= y1:
            return None

        crop = frame[y1:y2, x1:x2]
        ch, cw = crop.shape[:2]
        if ch < 20 or cw < 10:
            return None

        # Forma bölgesi: kutunun üst-orta kısmı
        yy1 = int(ch * 0.18)
        yy2 = int(ch * 0.58)
        xx1 = int(cw * 0.18)
        xx2 = int(cw * 0.82)
        jersey = crop[yy1:yy2, xx1:xx2]
        if jersey.size == 0:
            return None

        hsv = cv2.cvtColor(jersey, cv2.COLOR_BGR2HSV)
        # Çok karanlık / çok soluk pikselleri azalt
        mask = (hsv[:, :, 1] > 35) & (hsv[:, :, 2] > 40)
        if mask.sum() < 20:
            pixels_hsv = hsv.reshape(-1, 3)
            pixels_bgr = jersey.reshape(-1, 3)
        else:
            pixels_hsv = hsv[mask]
            pixels_bgr = jersey[mask]

        med_hsv = np.median(pixels_hsv, axis=0).astype(np.float32)
        mean_bgr = np.mean(pixels_bgr, axis=0).astype(np.float32)
        # Ölçekle: H 0-179, S/V/BGR 0-255
        feat = np.array([
            med_hsv[0] / 179.0,
            med_hsv[1] / 255.0,
            med_hsv[2] / 255.0,
            mean_bgr[0] / 255.0,
            mean_bgr[1] / 255.0,
            mean_bgr[2] / 255.0,
        ], dtype=np.float32)
        return feat

    def add_sample(self, feature: Optional[np.ndarray]):
        if not self.enabled or feature is None:
            return
        self.samples.append(feature)
        if len(self.samples) > TEAM_MAX_SAMPLES:
            self.samples = self.samples[-TEAM_MAX_SAMPLES:]

        enough = len(self.samples) >= TEAM_MIN_SAMPLES
        due = (len(self.samples) - self.last_fit_count) >= TEAM_REFIT_EVERY
        if enough and (self.kmeans is None or due):
            self.fit()

    def fit(self):
        if not self.enabled or len(self.samples) < TEAM_MIN_SAMPLES:
            return
        X = np.stack(self.samples, axis=0)
        self.kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
        self.kmeans.fit(X)
        self.last_fit_count = len(self.samples)

    def predict(self, feature: Optional[np.ndarray]) -> str:
        if not self.enabled or self.kmeans is None or feature is None:
            return UNKNOWN_TEAM_NAME
        pred = int(self.kmeans.predict(feature.reshape(1, -1))[0])
        return TEAM_1_NAME if pred == 0 else TEAM_2_NAME


def update_track_team(track: Track, team_label: str):
    if team_label not in {TEAM_1_NAME, TEAM_2_NAME}:
        return
    track.team_votes[team_label] += 1
    label, votes = track.team_votes.most_common(1)[0]
    if votes >= TEAM_MIN_TRACK_VOTES:
        track.team_label = label


def color_for_track(track: Track) -> Tuple[int, int, int]:
    if track.class_name == "referee":
        return REFEREE_COLOR
    if track.class_name == "goalkeeper":
        return GOALKEEPER_COLOR
    if track.team_label == TEAM_1_NAME:
        return TEAM_1_DRAW_COLOR
    if track.team_label == TEAM_2_NAME:
        return TEAM_2_DRAW_COLOR
    return UNKNOWN_PLAYER_COLOR


def draw_label(frame: np.ndarray, text: str, x: int, y: int, color: Tuple[int, int, int]):
    if not text:
        return
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.55
    thickness = 2
    (tw, th), baseline = cv2.getTextSize(text, font, scale, thickness)
    x = max(0, min(frame.shape[1] - tw - 2, x))
    y = max(th + 3, y)
    cv2.rectangle(frame, (x, y - th - baseline - 4), (x + tw + 6, y + baseline + 2), color, -1)
    cv2.putText(frame, text, (x + 3, y - 3), font, scale, TEXT_COLOR, thickness, cv2.LINE_AA)


def draw_track(frame: np.ndarray, track: Track):
    x1, y1, x2, y2 = track.bbox.astype(int)
    color = color_for_track(track)
    w = max(4, x2 - x1)
    cx = int((x1 + x2) / 2)
    foot_y = int(y2)

    if DRAW_BOXES:
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    if DRAW_ELLIPSES and track.class_name in {"player", "goalkeeper", "referee"}:
        axes = (max(10, int(w * 0.46)), max(4, int(w * 0.12)))
        cv2.ellipse(frame, (cx, foot_y), axes, 0, 0, 360, color, 3)

    parts = []
    if SHOW_TRACK_ID:
        parts.append(f"ID {track.track_id}")
    if SHOW_CLASS_NAME:
        parts.append(track.class_name)
    if SHOW_TEAM_LABEL and track.team_label != UNKNOWN_TEAM_NAME:
        parts.append(track.team_label.replace("team_", "T"))
    if SHOW_CONFIDENCE:
        parts.append(f"{track.confidence:.2f}")
    draw_label(frame, " | ".join(parts), x1, max(0, y1 - 8), color)


def draw_ball(frame: np.ndarray, det: Dict[str, Any]):
    x1, y1, x2, y2 = det["bbox"].astype(int)
    cx = int((x1 + x2) / 2)
    cy = int((y1 + y2) / 2)
    radius = max(5, int(max(x2 - x1, y2 - y1) / 2))
    cv2.circle(frame, (cx, cy), radius, BALL_COLOR, 2)

    if DRAW_BALL_TRIANGLE:
        top = (cx, max(0, y1 - 18))
        left = (cx - 10, max(0, y1 - 4))
        right = (cx + 10, max(0, y1 - 4))
        pts = np.array([top, left, right], dtype=np.int32)
        cv2.fillPoly(frame, [pts], BALL_COLOR)

    if SHOW_CLASS_NAME or SHOW_CONFIDENCE:
        label = "ball"
        if SHOW_CONFIDENCE:
            label += f" {det['confidence']:.2f}"
        draw_label(frame, label, x1, max(0, y1 - 22), BALL_COLOR)

print("Tracking / takım ayrımı / çizim fonksiyonları hazır.")


In [ ]:
# ============================================================
# 4) Videoyu işle
# ============================================================

def process_video(video_path: str) -> Tuple[str, str, Dict[str, Any]]:
    video_path = str(video_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Video açılamadı: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 1 or math.isnan(fps):
        fps = 25.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    if MAX_SECONDS is not None:
        max_frames = min(total_frames if total_frames > 0 else int(fps * MAX_SECONDS), int(fps * MAX_SECONDS))
    else:
        max_frames = total_frames if total_frames > 0 else None

    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    stem = Path(video_path).stem
    out_video = str(Path(OUTPUT_DIR) / f"{stem}_stable_tracking.mp4")
    out_csv = str(Path(OUTPUT_DIR) / f"{stem}_tracking.csv")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_video, fourcc, float(fps), (width, height))
    if not writer.isOpened():
        raise RuntimeError("VideoWriter açılamadı. Çıktı path veya codec sorunu olabilir.")

    rf = RoboflowRESTClient(ROBOFLOW_API_KEY, MODEL_ID, API_URL)
    tracker = SimpleStableTracker(fps=fps, max_missed_seconds=MAX_MISSED_SECONDS, match_distance_px=MATCH_DISTANCE_PX)
    teamer = JerseyTeamClassifier()

    records = []
    last_detections: List[Dict[str, Any]] = []
    last_ball_detections: List[Dict[str, Any]] = []
    processed = 0
    detection_calls = 0
    first_inference_ok = False

    pbar_total = max_frames if max_frames is not None else total_frames if total_frames > 0 else None
    pbar = tqdm(total=pbar_total, desc="Video işleniyor")

    frame_idx = 0
    while True:
        if max_frames is not None and frame_idx >= max_frames:
            break

        ok, frame = cap.read()
        if not ok:
            break

        annotated = frame.copy()
        do_detect = (frame_idx % max(1, int(DETECT_EVERY_N_FRAMES)) == 0)

        if do_detect:
            try:
                # Minimum eşik olarak en düşük class eşiği gönderiliyor; sınıf bazlı filtreyi parse aşamasında yapıyoruz.
                min_conf = min(PLAYER_CONFIDENCE, GOALKEEPER_CONFIDENCE, REFEREE_CONFIDENCE, BALL_CONFIDENCE)
                result, sx, sy = rf.infer_frame(frame, confidence=min_conf, overlap=OVERLAP)
                detections = parse_roboflow_predictions(result, scale_x=sx, scale_y=sy)
                last_detections = detections
                last_ball_detections = [d for d in detections if d["class"] == "ball"]
                detection_calls += 1
                first_inference_ok = True
            except Exception as e:
                # İlk frame'de bile API çalışmazsa net hata verelim.
                if not first_inference_ok:
                    writer.release()
                    cap.release()
                    raise RuntimeError(f"İlk Roboflow inference başarısız oldu: {repr(e)}")
                # Sonraki geçici hatalarda son detection'ı kullanarak devam et.
                print(f"Frame {frame_idx}: Roboflow geçici hata, son tespitler kullanılacak -> {repr(e)}")
                detections = last_detections
        else:
            detections = last_detections

        tracks = tracker.update(detections, frame_idx)

        # Takım ayrımı: sadece gerçek player için örnek topla.
        for tr in tracks:
            should_classify = tr.class_name == "player" or (CLASSIFY_GOALKEEPERS and tr.class_name == "goalkeeper")
            if should_classify:
                feat = teamer.extract_feature(frame, tr.bbox)
                teamer.add_sample(feat)
                label = teamer.predict(feat)
                update_track_team(tr, label)

        # Çizimler
        for ball in last_ball_detections:
            draw_ball(annotated, ball)

        for tr in tracks:
            draw_track(annotated, tr)

        # Sol üst bilgi paneli
        info = f"frame={frame_idx}  tracks={len(tracks)}  api_calls={detection_calls}"
        cv2.rectangle(annotated, (8, 8), (8 + 520, 42), (0, 0, 0), -1)
        cv2.putText(annotated, info, (16, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

        writer.write(annotated)

        # CSV kayıtları
        t_sec = frame_idx / float(fps)
        for tr in tracks:
            x1, y1, x2, y2 = tr.bbox.astype(float)
            fx, fy = foot_point(tr.bbox)
            records.append({
                "frame": frame_idx,
                "time_sec": round(t_sec, 4),
                "track_id": tr.track_id,
                "stable_id": tr.track_id,
                "class_name": tr.class_name,
                "confidence": round(float(tr.confidence), 5),
                "team_label": tr.team_label,
                "x1": round(x1, 2),
                "y1": round(y1, 2),
                "x2": round(x2, 2),
                "y2": round(y2, 2),
                "foot_x": round(fx, 2),
                "foot_y": round(fy, 2),
                "missed": tr.missed,
            })

        frame_idx += 1
        processed += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    writer.release()

    df = pd.DataFrame(records)
    df.to_csv(out_csv, index=False)

    summary = {
        "input_video": video_path,
        "output_video": out_video,
        "output_csv": out_csv,
        "fps": fps,
        "resolution": f"{width}x{height}",
        "processed_frames": processed,
        "detection_calls": detection_calls,
        "roboflow_method": rf.chosen_method,
        "csv_rows": len(df),
    }
    return out_video, out_csv, summary

video_file = choose_video_path(VIDEO_PATH)
print("Seçilen video:", video_file)

out_video, out_csv, summary = process_video(video_file)
print("\nİşlem tamamlandı.")
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 5) Çıktıyı önizle ve indir
# ============================================================
from pathlib import Path

print("Video çıktısı:", out_video)
print("CSV çıktısı:", out_csv)

# Colab içinde küçük önizleme
try:
    video_bytes = Path(out_video).read_bytes()
    video_b64 = base64.b64encode(video_bytes).decode("utf-8")
    display(HTML(f"""
    <video width="900" controls>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
    </video>
    """))
except Exception as e:
    print("Önizleme gösterilemedi:", repr(e))

# Colab download butonları
try:
    from google.colab import files
    files.download(out_video)
    files.download(out_csv)
except Exception as e:
    print("Colab download otomatik çalışmadı. Dosyalar burada:")
    print(out_video)
    print(out_csv)


## Sorun yaşarsan hızlı ayarlar

- Oyuncular kaçıyorsa: `PLAYER_CONFIDENCE = 0.15` veya `0.20` yap.
- Top az bulunuyorsa: `BALL_CONFIDENCE = 0.05` yap.
- ID çok değişiyorsa: `MATCH_DISTANCE_PX = 180` ve `MAX_MISSED_SECONDS = 1.5` deneyebilirsin.
- Çok yavaşsa: `DETECT_EVERY_N_FRAMES = 2` veya `3` yap; ama en stabil sonuç `1` değeridir.
- Tüm videoyu işlemek için: `MAX_SECONDS = None` yap.
